## 09 Human In the Loop (HTIL)

It is often the case that agents will need human intervention to resolve certain issues. This can be achieved via Human-in-the-loop (or HTIL). In LangChain agents, when defining an agent, you can specify for which tools calls you'll need human feedback. When one of those tools is called, an _interrupt_ is raised, asking for a human response. You can setup various _allowed_ responses, such as approvals, rejections and edits.



In [ ]:
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from dataclasses import dataclass

import langchain
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_community.utilities import SQLDatabase

print(f"Using langchain version: {langchain.__version__}")

load_dotenv(override=True)
console = Console()

Using langchain version: 1.2.14


**Step 1:**

As a first step, let's load the `Chinook` database and it's schema. The schema will help our agent generate the query faster.  

Next we define the runtime context for the LLM to hold the database connection as well as the schema information so that the agent is able to minimize loops before generating text-to-SQL. 

We also define our `execute_sql` tool and the system prompt - this time, we follow the simple modification of system prompt rather than use dynamic prompting with the `@dynamic_prompt` that we used in the [previous notebook](08_middleware.ipynb).

In [ ]:
from langchain_community.utilities import SQLDatabase
from typing import TypedDict, Optional, List
from pydantic import BaseModel

from langchain.tools import tool
from langgraph.runtime import get_runtime

# connect to our database -> in path db/chinook.db
db = SQLDatabase.from_uri("sqlite:///db/chinook.db")
schema = db.get_table_info()


# define our runtime context - we'll pass in the schema this time
class RuntimeContext(TypedDict):
    db: SQLDatabase
    db_schema: str


# define our tool to execute sql
@tool
def execute_sql(query: str) -> str:
    """execute query provided by user
    Args:
        query (str): SQL query to execute
    Returns:
        str: result of the query execution or error message
    """
    # get instance of db in context
    db: SQLDatabase = get_runtime().context["db"]
    try:
        result = db.run(query)
    except Exception as e:
        return f"Error occurred while executing SQL query: {e}"
    return str(result)


# define our system prompt with schema injected
SYSTEM_PROMPT = (
    """You are a careful SQLite Analyst.

Rules:
- Always think step-by-step
- When you need data, call the tool 'execute_sql' with ONE select query
- Read-only only; NO INSERT/UPDATE/DELETE/DROP/CREATE/REPLACE/TRUNCATE/ALTER
- Limit to 5 rows at the output, unless the user explicitly asks for more
- If the tool returns "Error:", revise the SQL and try again
- Prefer explicit column list, avoid SELECT *
- Here is the database schema you can refer to to generate the SQL
{database_schema}
"""
).format(database_schema=schema)

**Step 2**

HTIL are interrupts you define on certain tools that the agent has access to - you can choose which tools need the HTIL interrupt. This interrupt is "fired" AFTER the tool call, but BEFORE the LLM sees output of the tool. The internal Agent loop is interrupted at this point, the graph state is saved in LangGraph memory and control shifts to the caller of the agent (i.e. the Human). When we define the HTIL for an Agent, we specify what actions the Human is allowed to take - it can be one of ["accept", "reject", "edit"]

Next we create our agent where we specify our HTIL interrupt as follows:

In [ ]:
# define our agent
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
    context_schema=RuntimeContext,
    middleware=[
        HumanInTheLoopMiddleware(
            # interrup the flow AFTER return from execute_sql, but BEFORE the LLM
            # gets to see it. User is allowed to either "approve" or "reject"
            # there is another possible option "edit", which we have not used here!
            interrupt_on={"execute_sql": {"allowed_decisions": ["approve", "reject"]}},
        ),
        # define additional HTIL middleware instance if you have more tools
    ],
)

**Step 3**

Here is how we call the agent - 

In [14]:
from langgraph.types import Command

question = "What are the names of all the employees?"

config = {"configurable": {"thread_id": "1"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    context=RuntimeContext(db=db),
)

if "__interrupt__" in result:
    description = result["__interrupt__"][-1].value["action_requests"][-1][
        "description"
    ]
    print(f"\033[1;3;31m{80 * '-'}\033[0m")
    print(f"\033[1;3;31m Interrupt:{description}\033[0m")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [{"type": "reject", "message": "the database is offline."}]
            }
        ),
        config=config,  # Same thread ID to resume the paused conversation
        context=RuntimeContext(db=db),
    )
    print(f"\033[1;3;31m {result}\033[0m")
    print(f"\033[1;3;31m{80 * '-'}\033[0m")

print(result["messages"][-1].content)

--------------------------------------------------------------------------------
 Interrupt:Tool execution requires approval

Tool: execute_sql
Args: {'query': 'SELECT EmployeeId, FirstName, LastName FROM employees ORDER BY EmployeeId LIMIT 5;'}
 {'messages': [HumanMessage(content='What are the names of all the employees?', additional_kwargs={}, response_metadata={}, id='ef0dda86-4a89-43d3-b1df-f39096831602'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 297, 'prompt_tokens': 2350, 'total_tokens': 2647, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1664}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DT53Y5BO0JfDSC7Vchanf3WtL3yWu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs

In [15]:
print(result["messages"][-1])

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 358, 'prompt_tokens': 2401, 'total_tokens': 2759, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1664}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DT53f676p5f6DlNyOSKUPDbElIU08', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d7751-8e0e-7521-afa3-17ff55d60fcc-0' tool_calls=[{'name': 'execute_sql', 'args': {'query': 'SELECT FirstName, LastName FROM employees ORDER BY EmployeeId LIMIT 5;'}, 'id': 'call_LNcLd0MMse3A7Ui4uy3fcSve', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 2401, 'output_tokens': 358, 'total_tokens': 2759, 'input_token_details': {'audio': 0, 'cache_read': 1664}, 'output_token_

In [16]:
user_query = "List all my customers living in the city of London or Paris"

for step in agent.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    context=RuntimeContext(db=db, db_schema=schema),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

Notice the following message in the output

It shows that our decorated function `dynamic_system_prompt` is being called before it is passed to the model. 

> **⚠️ NOTE**
>
> Ok, this example is a bit contrived. In this specific case, we need not go through such an elaborate mechanism to modify the system prompt. Since we have already connected to our database & have our scheme _before_ we call our agent, we could have easily modified our system prompt with simple formatting.

Imagine a more realistic case: Let's say this is big corporate Sales database (instead of our Chinook database). Further assume we have a chat interface to this database and both internal employees as well as external "contracted staff" are able to use this chat interface to query the database for updates etc. Further imagine that we have a certain set of tables in our database that we **don't** want our contractors to query - say these are called: `customers_prospects`,` deals_early_stage` are two such tables. So if an employee were to ask _"List out the recent deals for prospect customer Slurpy Cones that our team is working on"_. In this case, the chat interface should respond (assume SQL query to this question comes from a join of these 2 tables). However, if an extranal contractor tries to query these tables, it should decline.

Here's how we could accomplish this:

1. Modify the runtime context like this
    
    ```python
    class RuntimeContext(TypedDict):
        db: SQLDatabase
        db_schema: str
        # is_employee is True for employees, False for contractor IDs
        is_employee: bool = True
    ```
2. Modify the `dynamic_system_prompt` function like this - here we are adding a line for non-employees to exclude the customers_prospects and deals_early_stage tables

    ```python
    @dynamic_prompt
    def dynamic_system_prompt(request: ModelRequest) -> str:
        # here we will format SYSTEM_PROMPT with actual value of db schema
        print("------ dynamic_system_prompt() called ---------")
        # grab the database schemd from the runtime context
        db_schema = request.runtime.context["db_schema"]
        is_employee = request.runtime.context["is_employee"]
        schema = db_schema
        if not is_employee:
            schema = schema + "\n Exclude the customers_prospects and deals_early_stage tables when generating the SQL!"
        return SYSTEM_PROMPT.format(database_schema=db_schema)
    ```
3. Here is how we would run a query on this agent
    ```python
    user_query = "List out the recent deals for prospect customer Slurpy Cones that our team is working on"

    for step in agent.stream(
        {"messages": [{"role": "user", "content": user_query}]},
        context=RuntimeContext(db=db, db_schema=schema, is_employee=False),
        stream_mode="values",
    ):
        step["messages"][-1].pretty_print()
    ```


### Filtering out PII information

Notice that I have received neatly parsed list of dicts - one for each row of data returned by the SQL query. The fields are parsed per the definition provided in the `response_format=Customers` parameter to the agent.

However, PII elements such as phone number and email are _clearly_ visible in the output. The objective of this example is to filter them out - more specifically REDACT then. Guess what? LangChain's middleware comes to our rescue again! Specifically the `PIIMiddleware` class.

LangChain provides a `PIIMiddleware` class in the `langchain.agents.middleware` package. It also provides some pre-defined filters for PII elements such as `email`, `credit cards` etc. For the full list see [API reference](https://reference.langchain.com/python/langchain/agents/middleware/pii/PIIMiddleware?_gl=1*q6e8ks*_gcl_au*MzAzOTE5NjkxLjE3NzQzNTU0MjM.*_ga*MzM4MDk1Mjk2LjE3NTE0NjQ5MDU.*_ga_47WX3HKKY2*czE3NzU2NDAxNjEkbzUxJGcxJHQxNzc1NjQzMzY5JGo1NSRsMCRoMA..)

Not all PII elements are pre-defined. For example, there is no pre-defined filter for phone numbers or names. However, we can define custom PII filters using reg-expressins for phone numbers. We'll address names a bit later.

Following cell shows how to use pre-defined `PIIMiddleware` for email and a custom `PIIMiddleware` for phone numbers. 
* Notice that in both these calls, we ask the middleware to REDACT (`strategy="redact"`) the information - instead of the field value, it will be displayed as "REDACTED_FieldName" in output. 
* The `apply_to_input=True` means _check_ (or fire) before model is called (this is default behavior).
There is also a `apply_to_output=True|False` parameter (False by default) that we have not used. Setting it to `True` will fire this middleware after LLM generates it's output. 
* The `apply_to_tool_results=True` means, apply this filter after the tool call (`execute_sql`) and _before_ results of the tool call go to the LLM.

LangChain provides some pre-defined filters -> `pii_type=Literal['email', 'credit_card', 'ip', 'mac_address', 'url']`.

We have also defined a custom `PIIMiddlware` filter for the phone number. We have provided one large reg-expression to check value. This works on the "phone" field and provided a `detector=PHONE_REGEX` to detect phone number pattern and REDACT the value, should the value in the phone field match the reg-ex pattern defined.

In [ ]:
from langchain.agents.middleware import PIIMiddleware

# email detector - in-built
email_detector = PIIMiddleware(
    pii_type="email",
    strategy="redact",
    apply_to_input=False,
    apply_to_tool_results=True,
)

# phone number - no in-built detector, so we use a regex

# Comprehensive pattern for North America, LATAM, Europe, Asia, and Africa
PHONE_REGEX = (
    r"(?:"
    # 1. International E.164 style (Starts with +)
    # Covers Europe, Asia, Africa, LATAM (+44, +91, +234, +49, etc.)
    r"\+\d{1,3}[\s\-\.]?\(?\d{1,4}\)?(?:[\s\-\.]\d{1,5}){1,4}"
    r"|"
    # 2. North American Numbering Plan (US/Canada)
    # Matches (555) 123-4567, 555-123-4567, 555.123.4567
    r"\(?\d{3}\)?[\s\-\.]\d{3}[\s\-\.]\d{4}"
    r"|"
    # 3. Generic Local Long Form (8-13 digits with separators)
    # Catches local formats in China, Japan, and parts of Europe
    r"(?:\b|(?<=\s))\d{2,4}[\s\-\.]\d{3,4}[\s\-\.]\d{3,4}\b"
    r")"
)


phone_detector = PIIMiddleware(
    "phone",
    # Reg-ex to detect
    detector=PHONE_REGEX,
    strategy="redact",
    apply_to_input=False,
    apply_to_tool_results=True,
)

Here is how we _attach_ the pre-defined and custom `PIIMiddleware` to our agent. Following is the definition of our `pii_agent`, which is _powered_ with `PIIMiddleware` objects to filter our email and phone number in addition to our `dynamic_system_prompt` middleware defined to specifically modify the system prompt.

In [ ]:
# now let's modify our agent definition
from langchain.agents import create_agent
from typing import List

pii_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[execute_sql],
    context_schema=RuntimeContext,
    response_format=Customers,
    middleware=[
        dynamic_system_prompt,
        email_detector,
        phone_detector,
    ],
)

Next, we run the same query as before, but now with our `pii_agent`. 

In [ ]:
# now let's ask the same query to the pii_agent
user_query = "List all my customers living in the city of London or Paris"

for step in pii_agent.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    context=RuntimeContext(db=db, db_schema=schema),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

List all my customers living in the city of London or Paris
------ dynamic_system_prompt() called ---------
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_ifkKXx7S3TDZfXsVONWB9sKc)
 Call ID: call_ifkKXx7S3TDZfXsVONWB9sKc
  Args:
    query: SELECT CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email
FROM customers
WHERE City IN ('London', 'Paris')
LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(39, 'Camille', 'Bernard', None, '4, Rue Milton', 'Paris', None, 'France', '75009', '+33 01 49 70 65 65', None, 'camille.bernard@yahoo.fr'), (40, 'Dominique', 'Lefebvre', None, '8, Rue Hanovre', 'Paris', None, 'France', '75002', '+33 01 47 42 71 71', None, 'dominiquelefebvre@gmail.com'), (52, 'Emma', 'Jones', None, '202 Hoxton Street', 'Lond

Notice that the structure of the output is the same as before, but the contents of the phone & email fields have been REDACTED - displaye [REDACTED_PHONE] and [REDACTED_EMAIL] where the values were displayed previously.


### Redacting the First and Last Names
By adding `PIIMiddleware` filters for email & phone, we saw that the agent is able to redact this information. Since these fields follow some sort of pattern, we can detect them early and filter out the information. Doing this with free-text fields such as names & addresses is a challenge. 

To address such filtering, LangChain provides a `@after_model` decorator, which you can apply to a custom function that will be called _after_ model generated output but _before_ this output is sent to the user. Since our output is structured, it will be able in the response['structured_output`] field (as we saw in the [LLM with Structured Output](07_structured_output.ipynb) example). In this function we iterate over the fields and explicitly set values of `firstName` and `lastName` to `[REDACTED]`.

In [ ]:
from langchain.agents.middleware.types import after_model
from langchain.agents.middleware import AgentState

# from langchain.agents.middleware.types import Runtime
from langgraph.runtime import Runtime

REDACTED = "[REDACTED]"


@after_model
def redact_name_fields(state: AgentState, runtime: Runtime) -> dict | None:
    structured = state.get("structured_response")

    # if no output from model, return nothing!
    if not structured:
        return None

    # this call replaces values of firstName and lastName with [REDACTED]
    redacted_customers = [
        c.model_copy(update={"firstName": REDACTED, "lastName": REDACTED})
        for c in structured.customers
    ]
    return {"structured_response": Customers(customers=redacted_customers)}

In [ ]:
# define our agent, with new @after_model middleware
from langchain.agents import create_agent

pii_agent2 = create_agent(
    model="openai:gpt-5-mini",
    tools=[execute_sql],
    context_schema=RuntimeContext,
    response_format=Customers,
    middleware=[
        dynamic_system_prompt,
        email_detector,
        phone_detector,
        redact_name_fields,
    ],
)

In [ ]:
# execute the same query as before
user_query = "List all my customers living in the city of London or Paris"

final_step = None
for step in pii_agent2.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    context=RuntimeContext(db=db, db_schema=schema),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    final_step = step

# The @after_model middleware updates structured_response in the agent state,
# NOT the raw AI message text — so we must read structured_response to see
# the redacted firstName / lastName values.
if final_step and "structured_response" in final_step:
    print(
        "\n--- Final structured output (names redacted by @after_model middleware) ---"
    )
    print(final_step["structured_response"].model_dump_json(indent=2))

================================ Human Message =================================

List all my customers living in the city of London or Paris
------ dynamic_system_prompt() called ---------
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_UYFrkNfknTfHvkd5TCdNObmA)
 Call ID: call_UYFrkNfknTfHvkd5TCdNObmA
  Args:
    query: SELECT CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email
FROM customers
WHERE lower(City) IN ('london','paris')
LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(39, 'Camille', 'Bernard', None, '4, Rue Milton', 'Paris', None, 'France', '75009', '+33 01 49 70 65 65', None, 'camille.bernard@yahoo.fr'), (40, 'Dominique', 'Lefebvre', None, '8, Rue Hanovre', 'Paris', None, 'France', '75002', '+33 01 47 42 71 71', None, 'dominiquelefebvre@gmail.com'), (52, 'Emma', 'Jones', None, '202 Hoxton Street',

The `@after_model` decoratored function was above to redact the `firstName` and `lastName` fields!

### Conclusion
In this notebook we saw:
1. How the `@dynamic_prompt` prompt, which is specifically designed, to modify the system prompt _before_ it is passed to the model. Our example is a bit contrived - we could have easily just formatted the SYSTEM_PROMPT inline!
2. Then we saw how pre-defined `PIIMiddleware` can be used to redact emails and a custom `PIIMiddleware` can be used to redact phone numbers.
3. Finally, we say how the `@after_model` decorated function can be used to redact fileds, such as `firstName` and `lastName` from the final structured output.